# LAB | Whisper STT Implementation Lab

Author: Louise Plessis

Description:

## Step 1: Setting Up the Environment

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

Key loaded: True


In [1]:
import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(result.stdout[:100])

FileNotFoundError: [Errno 2] No such file or directory: 'ffmpeg'

## Step 2: Downloading Sample Meeting Audio

In [3]:
import os

filepath = "audio/california_moon_landing.mp3"

print("File exists:", os.path.exists(filepath))
print("File size (KB):", os.path.getsize(filepath) / 1024)

with open(filepath, "rb") as f:
    header = f.read(20)
print("First 20 bytes:", header)

File exists: True
File size (KB): 909.6240234375
First 20 bytes: b'\xff\xfb\x90\xc4\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


File exists: True
File size (KB): 909.6240234375
First 20 bytes: b'\xff\xfb\x90\xc4\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [4]:
import librosa

y, sr = librosa.load(filepath, sr=None)
duration = librosa.get_duration(y=y, sr=sr)

print(f"Sample rate: {sr} Hz")
print(f"Duration: {duration:.2f} seconds")
print(f"Number of samples: {len(y)}")

Sample rate: 44100 Hz
Duration: 86.20 seconds
Number of samples: 3801347


Sample rate: 44100 Hz
Duration: 86.20 seconds
Number of samples: 3801347

## Step 3: Basic Transcription (Without Chunking)

In [13]:
# Basic transcription using Whisper
from openai import OpenAI
import os

# Initialize the client (reads OPENAI_API_KEY from your environment automatically)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

filepath = "audio/california_moon_landing.mp3"

with open(filepath, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print("Transcribed text:")
print(transcription.text)

Transcribed text:
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all. And

Transcribed text:

It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all. And they did probe for some samples. And just what was going to come of that, I don't know. I'll be glad to read it.

In [ ]:
# Json response with segments and timestamps
with open(filepath, "rb") as audio_file:
    transcription_verbose = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json"
    )

print("Detected language:", transcription_verbose.language)
print("Duration:", transcription_verbose.duration)
print("\nFirst segment example:")
print(transcription_verbose.segments[0])

Detected language: english
Duration: 86.19000244140625

First segment example:
TranscriptionSegment(id=0, avg_logprob=-0.37382397055625916, compression_ratio=1.5822222232818604, end=3.559999942779541, no_speech_prob=0.0020829029381275177, seek=0, start=0.0, temperature=0.0, text=' It was rather interesting just to watch them gathering', tokens=[50364, 467, 390, 2831, 1880, 445, 281, 6858, 339, 552, 13519, 50542])


## Step 4: Transcription with Prompts 

In [ ]:
filepath = "audio/california_moon_landing.mp3"

prompt_context = (
    "This is a 1969 conversation about the Apollo moon landing. "
    "Topics include astronauts, rendezvous procedures, lunar surface exploration, "
    "and NASA space missions."
    "Speaker is from Weaverville, CA; she is a 72-year-old white woman with a high-school education. 
)

with open(filepath, "rb") as audio_file:
    transcription_prompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=prompt_context
    )

print("Prompted transcription:")
print(transcription_prompted.text)

Prompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around. What do they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused. That's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area when you think of it. They just gathered rocks and samples of soil and all, and they did probe 

Prompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around. What do they call it? Kangaroo walk? Something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused. That's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area when you think of it. They just gathered rocks and samples of soil and all, and they did probe for some samples. Just what's going to come of that, I don't know. I'd be glad to read it.

## Step 5: Transcription Without Prompts

In [15]:
## Step 5: Transcription Without Prompts 
filepath = "audio/california_moon_landing.mp3"

with open(filepath, "rb") as audio_file:
    transcription_unprompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
        # note: no prompt parameter passed
    )

print("Unprompted transcription:")
print(transcription_unprompted.text)

Unprompted transcription:
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and 

#### Prompted vs. Unprompted Transcription Comparison

**Prompt used:** "This is a 1969 conversation about the Apollo moon landing. Topics include
astronauts, rendezvous procedures, lunar surface exploration, and NASA space missions." "Speaker is from Weaverville, CA; she is a 72-year-old white woman with a high-school education. 

##### Key differences

| Location in audio | Unprompted | Prompted | Notes |
|---|---|---|---|
| "get back to ___" | **Maine** | **Earth** | Prompt corrected a clear mishearing — "Maine" makes no sense in context, "Earth" does. Strongest example of prompt improving accuracy. |
| Future plans question | "Do you imagine them **sending up ships**?" | "Do you imagine them **standing up**?" | Here the *unprompted* version is more coherent. The prompted version arguably got worse — "standing up" doesn't fit the context as well as "sending up ships" does for a question about future NASA missions. |
| Closing line | "**I'll** be glad to read it" | "**I'd** be glad to read it" | Minor, likely inconsequential (ambiguous audio, either could be correct). |
| Punctuation/pacing | Slightly more run-on, extra "So"/"And" at sentence starts | Slightly cleaner sentence breaks | Prompted version is easier to read but not more *accurate* per se. |

##### Takeaway
The prompt context helped in one clear case (Maine → Earth) but didn't uniformly improve results — it introduced a plausible-but-wrong word ("standing up") in a spot where the unprompted model got the more sensible answer ("sending up ships"). This suggests prompts bias the model toward domain vocabulary but can occasionally override a correct guess with a contextually "expected" one that doesn't actually fit. Prompting is a net positive but not a guarantee of accuracy on every segment.

## Step 6: Implementing Audio Chunking


In [ ]:
from pydub import AudioSegment
import os

filepath = "audio/california_moon_landing.mp3"
chunk_length_minutes = 10  

audio = AudioSegment.from_mp3(filepath)
duration_ms = len(audio)
chunk_length_ms = chunk_length_minutes * 60 * 1000

os.makedirs("audio/chunks", exist_ok=True)

chunks = []
for i, start_ms in enumerate(range(0, duration_ms, chunk_length_ms)):
    end_ms = min(start_ms + chunk_length_ms, duration_ms)
    chunk = audio[start_ms:end_ms]
    chunk_path = f"audio/chunks/chunk_{i}.mp3"
    chunk.export(chunk_path, format="mp3")
    chunks.append({
        "path": chunk_path,
        "start_ms": start_ms,
        "end_ms": end_ms
    })
    print(f"Created {chunk_path}: {start_ms/1000:.1f}s - {end_ms/1000:.1f}s")

print(f"\nTotal chunks created: {len(chunks)}")